In [0]:
bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
table = "country_code"
bronze_table_parquet_path = f"{bronze}/{table}"

country_code_df = spark.read.format("parquet")\
    .load(f"{bronze_table_parquet_path}")

display(country_code_df)

country_code,country_name,created_at,flag,id,date_type,year,month,day,_rescued_data
40,South Africa,2026-06-06T01:12:00.513Z,false,40,2026-06-06,2026,6,6,null
41,Kenya,2026-06-06T01:12:00.513Z,false,41,2026-06-06,2026,6,6,null
42,Tanzania,2026-06-06T01:12:00.513Z,false,42,2026-06-06,2026,6,6,null
43,Argentina,2026-06-06T01:12:00.513Z,false,43,2026-06-06,2026,6,6,null
44,Chile,2026-06-06T01:12:00.513Z,false,44,2026-06-06,2026,6,6,null
45,Peru,2026-06-06T01:12:00.513Z,false,45,2026-06-06,2026,6,6,null
46,Colombia,2026-06-06T01:12:00.513Z,false,46,2026-06-06,2026,6,6,null
47,Nepal,2026-06-06T01:12:00.513Z,false,47,2026-06-06,2026,6,6,null
48,Sri Lanka,2026-06-06T01:12:00.513Z,false,48,2026-06-06,2026,6,6,null
49,Cambodia,2026-06-06T01:12:00.513Z,false,49,2026-06-06,2026,6,6,null


## Quality

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, coalesce, when
from pyspark.sql.functions import col, coalesce, get_json_object, to_timestamp, lit, when


def transform_to_silver(bronze_df: DataFrame) -> DataFrame:
    df = bronze_df
    df = df.withColumn("created_at",
                               when(col("created_at").isNull(),
                                    get_json_object(col("_rescued_data"),"$.created_at"))
                                    .otherwise(col("created_at"))
                                    )
    
    df = (
        df
        # if the year is 2024, replace it with 2026 (keeps month/day/time exactly)
        .withColumn(
            "created_at",
            F.when(
                F.col("created_at").startswith("2024"),
                F.regexp_replace("created_at", r"^2024", "2026")
            ).otherwise(F.col("created_at"))
        )
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("date_type", F.to_date("created_at"))
    )
    # date_type is a real date; created_at is "-" so we skip it
    df = (
        df.withColumn("created_at", F.to_timestamp("created_at"))
          .withColumn("date_type", F.to_date("date_type"))
    )

    df = df.withColumn("flag", F.lower(F.trim(F.col("flag"))) == F.lit("true"))

    df = (
        df.withColumn("year",          F.expr("try_cast(year as int)"))
          .withColumn("month",         F.expr("try_cast(month as int)"))
          .withColumn("day",           F.expr("try_cast(day as int)"))
          .withColumn("id",            F.expr("try_cast(id as int)"))
          .withColumn("country_code", F.expr("try_cast(country_code as int)"))
    )

    df = df.withColumn(
        "_is_valid",
        coalesce(
            col("id").isNotNull()
            & col("country_code").isNotNull()
            & col("id").isNotNull(),
            lit(False),
        ),
    )

    valid_df   = df.filter(col("_is_valid"))
    invalid_df = df.filter(~col("_is_valid"))

    invalid_count = invalid_df.count()

    if invalid_count > 0:
        print(f"Quarantined {invalid_count} invalid records")

    return valid_df.drop("_rescued_data","_is_valid")


df = transform_to_silver(country_code_df)


## Deduplicated

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def deduplicate_by_key(df, keycolumns, order_column, ascending=False):
    """
    
    Deduplicate a Dataframe by composite key, keeping the row with the highest (or lowest) value in order_column

    Args:
        df: Input DataFrame with duplicates
        key_columns: List of columns forming the composite key
        order_column: Column to break ties (e.g., updated_at)
        ascending: If True, keep the smallest order_column value
    """
    
    order_expr = (
        F.col(order_column).asc() if ascending else F.col(order_column).desc()
    )

    window_spec = Window.partitionBy(*keycolumns).orderBy(order_expr)

    return df.withColumn("rank", F.row_number().over(window_spec)).filter(
        F.col("rank") == 1
    ).drop("rank")

country_code_df = deduplicate_by_key(df, ["id"], "created_at", ascending=True)



In [0]:
country_code_df.display()

country_code,country_name,created_at,flag,id,date_type,year,month,day
1,Thailand,2026-06-06T01:12:00.513Z,false,1,2026-06-06,2026,6,6
2,Japan,2026-06-06T01:12:00.513Z,false,2,2026-06-06,2026,6,6
3,Norway,2026-06-06T01:12:00.513Z,false,3,2026-06-06,2026,6,6
4,Sweden,2026-06-06T01:12:00.513Z,false,4,2026-06-06,2026,6,6
5,Austria,2026-06-06T01:12:00.513Z,false,5,2026-06-06,2026,6,6
6,Italy,2026-06-06T01:12:00.513Z,false,6,2026-06-06,2026,6,6
7,France,2026-06-06T01:12:00.513Z,false,7,2026-06-06,2026,6,6
8,Germany,2026-06-06T01:12:00.513Z,false,8,2026-06-06,2026,6,6
9,USA,2026-06-06T01:12:00.513Z,false,9,2026-06-06,2026,6,6
10,UK,2026-06-06T01:12:00.513Z,false,10,2026-06-06,2026,6,6


## Data Writing

In [0]:
country_code_df.write.format("delta").mode("overwrite").save("abfss://silver@databricktraveljournal.dfs.core.windows.net/country_code")

## Delta

##

In [0]:
%sql

CREATE TABLE IF NOT EXISTS travel_journal_catalog.silver.country_code 
USING DELTA
LOCATION "abfss://silver@databricktraveljournal.dfs.core.windows.net/country_code"